In [ ]:
pip install xgboost

In [ ]:
import pandas as pd
import numpy as np
import pickle
import platform
import pkg_resources
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

/var/folders/z6/vcd1nm395kx4c8bz82wk7hdw0000gn/T/ipykernel_3131/2277926219.py:5: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


In [2]:
# Document environment dependencies
# OS info
print("Operating system:", platform.system(), platform.release())

# Python version
print("Python version:", platform.python_version())

# Installed packages
packages = ['pandas', 'numpy', 'scikit-learn', 'xgboost']
for package in packages:
    version = pkg_resources.get_distribution(package).version
    print(f"{package}=={version}")


Operating system: Darwin 23.1.0
Python version: 3.12.5
pandas==2.2.3
numpy==1.26.4
scikit-learn==1.5.2
xgboost==2.1.4


In [ ]:
# Load training data
train_wide = pd.read_csv("../Data/week5_wide_data.csv")

# Define target and features
target_col = "Life expectancy at birth, total (years)"
drop_cols = ["country_name", "year", target_col]
X_full = train_wide.drop(columns=drop_cols)
y_train = train_wide[target_col]

# Prepare both original and bias-mitigated models input features
bias_cols = [col for col in X_full.columns if "GNI" in col or "health expenditure" in col]
object_cols = X_full.select_dtypes(include="object").columns

# for xgb_model which keep bias features
X_original = X_full.drop(columns=object_cols) 
# for xgb_mitigated which remove bias
X_mitigated = X_full.drop(columns=bias_cols + list(object_cols), errors="ignore")  

# Train original model
xgb_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    colsample_bytree=1.0,
    subsample=1.0,
    random_state=42
)
xgb_model.fit(X_original, y_train)

# Train bias-mitigated model
xgb_mitigated = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    colsample_bytree=1.0,
    subsample=1.0,
    random_state=42
)
xgb_mitigated.fit(X_mitigated, y_train)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=1.0, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=100, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

In [5]:
# Saving the original model for deployment as a pickle file
with open("xgb_model.pkl", "wb") as f:
    pickle.dump(xgb_model, f)

# Saving the bias mitigation model for deployment as a pickle file
with open("xgb_mitigated_model.pkl", "wb") as f:
    pickle.dump(xgb_mitigated, f)


In [6]:
# Predict training data for both models
y_pred_original = xgb_model.predict(X_original)
y_pred_mitigated = xgb_mitigated.predict(X_mitigated)

# Calculate RMSE
rmse_original = np.sqrt(mean_squared_error(y_train, y_pred_original))
rmse_mitigated = np.sqrt(mean_squared_error(y_train, y_pred_mitigated))

# Calculate R^2
r2_original = r2_score(y_train, y_pred_original)
r2_mitigated = r2_score(y_train, y_pred_mitigated)

# Results
print("Original model performance:")
print(f"RMSE: {rmse_original:.4f}")
print(f"R^2: {r2_original:.4f}")

print("\nBias-mitigated model performance:")
print(f"RMSE: {rmse_mitigated:.4f}")
print(f"R^2: {r2_mitigated:.4f}")


Original model performance:
RMSE: 0.0687
R^2: 0.9999

Bias-mitigated model performance:
RMSE: 0.0661
R^2: 0.9999


The bias-mitigated model achieved a slightly lower RMSE compared to the original model. Therefore, we selected the xgb_mitigated_model.pkl as the final model for deployment.

In [4]:
# Sample predictions using the original model with bias features
sample_preds_original = xgb_model.predict(X_original.sample(5, random_state=1))
print("Original model predictions:", sample_preds_original)

# Sample predictions using the bias-mitigated model
sample_preds_mitigated = xgb_mitigated.predict(X_mitigated.sample(5, random_state=1))
print("Bias-mitigated model predictions:", sample_preds_mitigated)


Original model predictions: [81.2948   74.619804 65.99316  63.689926 67.37547 ]
Bias-mitigated model predictions: [81.30875  74.63274  65.9738   63.717133 67.36198 ]


The comparison shows that removing bias-prone features had only a small effect on the model’s predictions. The differences were minimal. This suggests that the model was able to maintain strong predictive performance using other correlated indicators. The result is encouraging, as it confirms that bias mitigation did not significantly harm model accuracy, and still helping improve fairness across income groups. 

# Deployment mode: batch inference
We chose to deploy the model using batch inference instead of real-time prediction. This makes more sense for our use case because the data we're working with is updated on a yearly basis, not in real time. Since the goal of the model is to support long-term planning and policy decisions, there's no need to generate predictions on-demand.

Real-time inference would add unnecessary complexity without offering much benefit, because the data just doesn't change that frequently. On the other hand, with batch processing, we can run the model once new data becomes available, generate predictions for all countries at once, and use those results to guide healthcare strategy. It's a more efficient and practical choice for the type of problem we're solving.

# Performance metrics for monitoring 
To make sure the model remains accurate, fair, and reliable over time, we plan to track:
1. Model performance metrics like RMSE and R^2  on new incoming data to monitor prediction accuracy and model stability. These metrics help us detect if the model's overall performance drops due to changes in data patterns or real-world conditions.
2. Data quality metrics such as missing values, outliers, or feature distribution shifts. These checks are important because poor data quality can cause the model to behave unpredictably even if the model itself hasn't changed.
3. Fairness and bias metrics, specifically RMSE broken down by income group (low, middle, high), to check if the model performs consistently across different segments. This matters especially in our case since the project is focused on healthcare equity.
Tracking these metrics gives us early visibility into problems, such as those caused by data changes, environmental disruptions, or fairness concerns, so we can respond before the model's performance meaningfully degrades.

# Monitoring thresholds
To make sure our model stays accurate, fair, and trustworthy in production, we’ve defined thresholds for key performance metrics based on our actual validation and test results. They help us track when the model is working well, when caution is needed, and when action must be taken.
1. RMSE (prediction error)
    - Green (RMSE ≤ 0.50): this is in line with our current test RMSE (0.36), so anything under 0.50 indicates the model is performing as expected.
    - Yellow (0.50 < RMSE ≤ 1.0): a moderate increase suggests potential issues like data drift or changes in input patterns. We’ll investigate further and monitor more frequently.
    - Red (RMSE > 1.0): this is a major deviation from baseline performance. The model may no longer be reliable and should be pulled from production for retraining.
2.  R^2 (explained variance)
    - Green (R² ≥ 0.98): consistent with our current validation and test performance. No action needed.
    - Yellow (0.95 ≤ R² < 0.98): a slight drop signals the model is starting to lose predictive power. Requires investigation.
    - Red (R² < 0.95): a strong sign that the model is not explaining enough variation and may need retraining.
3. Fairness (RMSE gap between income groups)
    - Green (gap ≤ ±0.15): predictive error is balanced across income levels.
    - Yellow (gap between 0.15 and 0.40): indicates emerging disparities in model performance for certain groups.
    - Red (gap > 0.40): a large fairness gap suggests systemic bias. The model should be reviewed and adjusted.

# Risk mitigation plan by threshold level
1. Green:
    - No immediate action needed.
    - Continue regular monitoring and logging.
    - Archive outputs for periodic audits and retraining benchmarks.
2. Yellow:
    - Investigate potential causes like data drift, input quality, or domain shifts.
    - Increase monitoring frequency.
    - Consider updating features, retraining the model, or re-tuning hyperparameters.
3. Red:
    - Pull model from production to prevent unreliable outputs. 
    - Retrain using the latest and cleaned data.
    - Reassess feature relevance and fairness across groups and bias.
    - Notify relevant stakeholders if the model influences real-world decisions.

These thresholds are grounded in our model’s past performance and the range of our target variable, which allows enough flexibility to account for natural variation while still flagging potential issues early. With defined limits and response plans, we can stay ahead of performance problems, reduce bias risk, and keep the model reliable in production.

# Discuss if and how frequently you would retrain your model and why?
Given that our project uses global healthcare indicators such as life expectancy, mortality rates, and health expenditures, which typically evolve slowly over time, we recommend retraining the model once per year. Annual retraining aligns with the release cycle of updated datasets from sources like the World Bank. This make sure that the model stays accurate as countries improve or worsen in healthcare access and outcomes. Retraining more frequently isn’t necessary unless there are major external shocks, such as a global pandemic, large-scale conflict, or major economic shifts that could rapidly alter healthcare systems. In those cases, early retraining would be triggered to keep the model responsive to new realities.


# Discuss if the data for your use case can be impacted by data drift, or concept drift and how you’d mitigate these risks
Our model could be affected by data drift, where the distribution of input features like health expenditure or adult mortality rates shifts gradually due to evolving global conditions. Concept drift is less immediate but still possible over time if the underlying relationship between predictors and life expectancy changes. For instance, if new healthcare technologies make certain indicators less predictive. To mitigate these risks, we will monitor key performance metrics like RMSE, R^2, and fairness gaps across income groups after each batch prediction cycle. We have also set clear thresholds for flagging model degradation, and if a yellow or red alert is triggered, retraining will be prioritized. This approach make sure the model remains fair and relevant as real-world healthcare dynamics change.